# Model Training

In [ ]:
# Set to True to persist data/checkpoints/logs in Google Drive across sessions
# (strongly recommended -- Colab's local disk is wiped whenever the runtime resets).
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/food_classifier"
else:
    BASE_DIR = "/content/food_classifier"

print(f"Using BASE_DIR = {BASE_DIR}")

## Import Python Modules

In [ ]:
import copy
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm  # tqdm.auto renders as a proper widget in Colab/Jupyter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
torch.manual_seed(42)

## Experiment Configuration

In [ ]:
@dataclass
class Config:
    # Merged dataset (Food-101 + UEC-256), rebuilt fresh each session -- see
    # section 3.5 below. Local disk is fine (and faster) since it's cheap to
    # regenerate from the Drive-cached sources every time.
    merged_data_dir: str = "/content/merged_food_dataset"

    # Sources for rebuilding the merged dataset. These should point at Drive
    # so the ~5GB Food-101 download, the cropped UEC images, and (most
    # importantly) hand-reviewed class_mapping.json all persist across
    # sessions instead of being lost when the Colab runtime resets.
    food101_source_dir: str = f"./data"
    uec_cropped_dir: str = f"{BASE_DIR}/data/uec256_cropped"
    class_mapping_path: str = f"{BASE_DIR}/class_mapping.json"
    uec_test_fraction: float = 0.15

    # Fraction of the TRAIN split carved off as a validation set, used to
    # pick the best checkpoint during training. Stratified per class so rare
    # classes don't end up with zero validation examples by bad luck. The
    # test set stays completely untouched until one final evaluation at the
    # end of training -- it's never used for model selection.
    val_fraction: float = 0.2
    split_seed: int = 42

    output_dir: str = f"{BASE_DIR}/checkpoints"
    logdir: str = f"{BASE_DIR}/runs"

    batch_size: int = 64
    num_workers: int = 2          # Colab notebooks are often happier with fewer workers than a script
    image_size: int = 320

    epochs_head: int = 3
    epochs_finetune: int = 10
    lr_head: float = 1e-3
    lr_finetune: float = 1e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    grad_clip_norm: float = 1.0
    warmup_epochs: int = 2         # linear LR warmup epochs before cosine annealing kicks in, per phase

    # Partial/gradual unfreezing for phase 2: only unfreeze from this ResNet
    # stage onward (see RESNET_STAGE_ORDER). "conv1" = unfreeze everything
    # (old full-unfreeze behavior). "layer3" leaves the lowest-level, most
    # generic features frozen, reducing how much phase 2 can overfit.
    finetune_unfreeze_from: str = "layer3"

    # MixUp/CutMix during phase 2 training only -- helps sharpen decision
    # boundaries between visually similar-but-distinct classes. Set
    # use_mixup=False to disable it.
    use_mixup: bool = True
    mixup_alpha: float = 0.2
    cutmix_alpha: float = 1.0
    mixup_prob: float = 0.5

    # Path to a "latest" checkpoint to resume from, or None to start fresh.
    # e.g. resume = f"{BASE_DIR}/checkpoints/resnet50_food_latest.pt"
    resume: str = f"{BASE_DIR}/checkpoints/resnet50_food_latest.pt"

    # Override the TensorBoard run folder name. Leave as None to auto-pick:
    # a fresh timestamp for a new run, or the resumed checkpoint's original
    # run name when `resume` is set (so curves continue in the same chart).
    run_name: str = f"{BASE_DIR}/runs/20260815_095304"


CFG = Config()

Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.logdir).mkdir(parents=True, exist_ok=True)

checkpoint_path = str(Path(CFG.output_dir) / "resnet50_food_best.pt")
latest_checkpoint_path = str(Path(CFG.output_dir) / "resnet50_food_latest.pt")
classes_path = str(Path(CFG.output_dir) / "classes.txt")

# NUM_CLASSES is no longer a fixed constant - with the merged dataset it
# depends on how many classes survived your class_mapping.json
# (315 at last count). It gets set for real once the merged
# dataset is loaded and its classes are known.
NUM_CLASSES = None
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(CFG)

## Build Merged Dataset

This assumes we've already downloaded the [UEC-256](http://foodcam.mobi/dataset256.html) dataset, run `prepare_uec256.py` (crop UEC-256 to bounding boxes) and
`build_class_mapping.py` (produced `class_mapping.json`) at least once, with
their outputs cached on Drive or local file system per `CFG.food101_source_dir` / `CFG.uec_cropped_dir` /
`CFG.class_mapping_path` above. Those are the slow, one-time steps.

This cell **copies** (not symlinks) Food-101 + UEC images into `CFG.merged_data_dir` on local
disk. Copying, rather than symlinking straight into Drive, is the important part: a symlink
into Drive still reads through Drive's slow FUSE mount on every access, and training re-reads
every image once per epoch -- so that latency gets paid over and over across the whole training
run instead of once. A local copy costs a few minutes up front here, but every epoch after that
reads from fast local disk instead. This cell needs to re-run every session since local Colab
disk is wiped on reset; it will loudly tell us if any expected source file is missing instead
of silently producing a broken/partial dataset.

In [ ]:
import json
import random
import re
import shutil

random.seed(42)


def slugify(name: str) -> str:
    """Must match the slugify used when the UEC images were cropped/mapped."""
    return re.sub(r"[^a-z0-9]+", "_", name.strip().lower()).strip("_")


def copy_image(src: Path, dst: Path):
    """
    Always copies (never symlinks) -- see the note above. A symlink pointing
    back into Drive would still pay Drive's slow per-file FUSE latency on
    every training read, every epoch, which is far more expensive overall
    than paying the copy cost once here.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    shutil.copy2(src, dst)


def rebuild_merged_dataset(cfg: Config):
    output_dir = Path(cfg.merged_data_dir)
    if output_dir.exists():
        print(f"{output_dir} already exists -- removing before rebuilding")
        shutil.rmtree(output_dir)

    # --- Food-101 (keeps its official train/test split) ---
    for split in ("train", "test"):
        ds = datasets.Food101(root=cfg.food101_source_dir, split=split, download=True)
        for img_path, label_idx in tqdm(list(zip(ds._image_files, ds._labels)),
                                         desc=f"copying Food-101 {split}"):
            class_name = ds.classes[label_idx]
            copy_image(Path(img_path), output_dir / split / class_name / Path(img_path).name)
        print(f"  Food-101 {split}: {len(ds._image_files)} images")

    # --- UEC-256 (random per-category split, no official one exists) ---
    mapping = json.loads(Path(cfg.class_mapping_path).read_text(encoding="utf-8"))
    uec_cropped_dir = Path(cfg.uec_cropped_dir)
    n_train, n_test, n_missing = 0, 0, 0
    for cat_id, info in tqdm(list(mapping.items()), desc="copying UEC-256"):
        final_class = info["final_class_name"]
        src_dir = uec_cropped_dir / slugify(info["uec_name"])
        if not src_dir.exists():
            n_missing += 1
            continue
        images = sorted(src_dir.glob("*.jpg"))
        random.shuffle(images)
        n_test_i = max(1, int(len(images) * cfg.uec_test_fraction))
        for img in images[n_test_i:]:
            copy_image(img, output_dir / "train" / final_class / img.name)
            n_train += 1
        for img in images[:n_test_i]:
            copy_image(img, output_dir / "test" / final_class / img.name)
            n_test += 1
    print(f"  UEC-256: {n_train} train images, {n_test} test images "
          f"({n_missing} categories had no cropped images -- check uec_cropped_dir if this is nonzero)")

    train_classes = sorted(p.name for p in (output_dir / "train").iterdir() if p.is_dir())
    test_classes = sorted(p.name for p in (output_dir / "test").iterdir() if p.is_dir())
    print(f"\nRebuilt {output_dir}: {len(train_classes)} train classes, {len(test_classes)} test classes")
    only_train = set(train_classes) - set(test_classes)
    only_test = set(test_classes) - set(train_classes)
    if only_train or only_test:
        print(f"  note: {len(only_train)} classes have zero test images, {len(only_test)} have zero train images ")


rebuild_merged_dataset(CFG)

## Data loading (merged Food-101 + UEC-256)

Loads `CFG.merged_data_dir` using a **shared class list built from the union of the train and
test folders**, rather than two independently-constructed `ImageFolder`s. This matters: if two
separate `ImageFolder`s each infer their own classes from whatever subfolders happen to be
present, a class that's missing from one split (e.g. a rare UEC category with very few images)
would silently get a *different* label index in train vs. test.
Building one canonical class list up front and reusing it for both splits rules that out.

Also carves a **validation set** out of the train split (`CFG.val_fraction`, stratified per
class) so there are three separate roles: **train** the model, use **val** to pick the best
checkpoint during training, and touch **test** exactly once at the very end for a final,
unbiased number. Using the test set for per-epoch model selection would quietly leak
information from it into training decisions - defeating the point of holding it out at all.

In [ ]:
def collect_class_samples(root, classes, class_to_idx, extensions=(".jpg", ".jpeg", ".png")):
    """Scans root/<class_name>/* for each class in the shared class list -> list of (path, label)."""
    samples = []
    for class_name in classes:
        class_dir = Path(root) / class_name
        if not class_dir.is_dir():
            continue  # this class just has 0 samples in this split
        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() in extensions:
                samples.append((img_path, class_to_idx[class_name]))
    return samples


def stratified_train_val_split(samples, val_fraction: float = 0.2, seed: int = 42):
    """
    Splits (path, label) samples into train/val, preserving each class's
    proportion via a manual per-class split.

    Deliberately NOT sklearn.model_selection.train_test_split(stratify=...)
    -- that raises immediately if ANY single class has fewer than 2
    members, which is a real risk here given how small some of the merged
    UEC categories can be. This does the best it can per class instead of
    failing the whole split over one rare class.

    Classes with < 2 samples keep everything in train (too few to split).
    Classes with >= 2 samples always keep at least 1 example in each split.
    """
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for sample in samples:
        by_class[sample[1]].append(sample)

    train_samples, val_samples = [], []
    n_too_few = 0
    for label, class_samples in by_class.items():
        class_samples = class_samples[:]
        rng.shuffle(class_samples)
        n = len(class_samples)
        if n < 2:
            train_samples.extend(class_samples)
            n_too_few += 1
            continue
        n_val = max(1, round(n * val_fraction))
        n_val = min(n_val, n - 1)  # always leave >= 1 example for train
        val_samples.extend(class_samples[:n_val])
        train_samples.extend(class_samples[n_val:])

    if n_too_few:
        print(f"  note: {n_too_few} classes had <2 training images -- kept entirely in train, "
              f"no validation representation for them")

    return train_samples, val_samples


class ImageSampleDataset(Dataset):
    """A Dataset over an explicit list of (path, label) samples with a given transform."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def get_dataloaders(data_dir: str, batch_size: int, num_workers: int, image_size: int = 224,
                     val_fraction: float = 0.2, split_seed: int = 42):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    # No augmentation for val/test -- both are purely for evaluation, so both
    # get the same deterministic resize/crop. They stay as separate loaders
    # so the test set is never touched during training or model selection.
    eval_transform = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_root = Path(data_dir) / "train"
    test_root = Path(data_dir) / "test"

    # Canonical class list = union of both splits, sorted for determinism --
    # shared everywhere below so label indices always agree.
    classes = sorted({p.name for p in train_root.iterdir() if p.is_dir()}
                      | {p.name for p in test_root.iterdir() if p.is_dir()})
    class_to_idx = {c: i for i, c in enumerate(classes)}

    all_train_samples = collect_class_samples(train_root, classes, class_to_idx)
    train_samples, val_samples = stratified_train_val_split(all_train_samples, val_fraction, split_seed)
    test_samples = collect_class_samples(test_root, classes, class_to_idx)

    train_set = ImageSampleDataset(train_samples, transform=train_transform)
    val_set = ImageSampleDataset(val_samples, transform=eval_transform)
    test_set = ImageSampleDataset(test_samples, transform=eval_transform)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader, classes


train_loader, val_loader, test_loader, class_names = get_dataloaders(
    CFG.merged_data_dir, CFG.batch_size, CFG.num_workers, CFG.image_size,
    CFG.val_fraction, CFG.split_seed,
)
NUM_CLASSES = len(class_names)
Path(classes_path).write_text("\n".join(class_names))
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | "
      f"Test: {len(test_loader.dataset)} | Classes: {NUM_CLASSES}")

## ResNet50 Classifier (Transfer Learning)

In [ ]:
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    # Dropout before the final Linear helps reduce overfitting, since
    # Food-101 is much smaller/noisier than ImageNet.
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


def set_backbone_trainable(model: nn.Module, trainable: bool):
    """Freezes or unfreezes every parameter except the final `fc` head."""
    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = trainable


RESNET_STAGE_ORDER = ["conv1", "bn1", "relu", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool", "fc"]


def set_layers_trainable(model: nn.Module, unfreeze_from: str = "layer3"):
    """
    Partial/gradual unfreezing: freezes everything before `unfreeze_from` in
    ResNet50's stage order, unfreezes `unfreeze_from` onward (plus the head,
    always trainable).

    The default "layer3" keeps conv1/bn1/layer1/layer2 -- the lowest-level,
    most generic visual features (edges, textures, colors) -- frozen even
    during "fine-tuning", while layer3/layer4 (higher-level, more
    task-specific features) adapt to food images. This gives most of full
    fine-tuning's benefit with meaningfully less capacity to overfit, since
    far fewer parameters are being updated. Use "conv1" for the old
    behavior (unfreeze everything).
    """
    if unfreeze_from not in RESNET_STAGE_ORDER:
        raise ValueError(f"unfreeze_from must be one of {RESNET_STAGE_ORDER}")
    trainable_stages = set(RESNET_STAGE_ORDER[RESNET_STAGE_ORDER.index(unfreeze_from):]) | {"fc"}

    for name, param in model.named_parameters():
        top_level_module = name.split(".")[0]
        param.requires_grad = top_level_module in trainable_stages


def freeze_batchnorm_stats(model: nn.Module):
    """
    Puts every BatchNorm layer whose weights are frozen into eval mode.
    `model.train()` puts BatchNorm into training mode regardless of
    requires_grad, so without this, a "frozen" backbone still shifts its
    running statistics batch to batch - a common cause of noisy loss
    during head-only training.
    """
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            if module.weight is not None and not module.weight.requires_grad:
                module.eval()


def build_scheduler(optimizer, total_epochs: int, warmup_epochs: int = 0):
    """
    Epoch-level LR schedule: `warmup_epochs` of linear warmup (ramping from
    10% of the optimizer's set LR up to 100%) followed by cosine annealing
    for the remaining epochs.

    Warmup mainly helps *training stability* early on - e.g. right after
    unfreezing the backbone in phase 2, when gradients hitting previously-
    frozen layers can be noisy/large.

    Falls back to plain cosine annealing if warmup_epochs <= 0, and clamps
    warmup_epochs to leave at least 1 epoch for the cosine phase so a short
    phase doesn't get entirely swallowed by warmup.
    """
    warmup_epochs = max(0, min(warmup_epochs, total_epochs - 1))
    if warmup_epochs == 0:
        return CosineAnnealingLR(optimizer, T_max=total_epochs)

    warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs)
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])


model = build_model(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
print(model.fc)

### Checkpointing (save/resume)

Two files get saved to `CFG.output_dir`:
- **`resnet50_food_best.pt`** — plain model weights, updated whenever validation accuracy improves. Used for inference.
- **`resnet50_food_latest.pt`** — full resumable state (model + optimizer + scheduler + phase/epoch/TensorBoard counters + run name), overwritten after every epoch. Used for `CFG.resume`.

In [ ]:
def save_checkpoint(path: str, model, optimizer, scheduler, phase: str, epoch: int,
                     global_epoch: int, global_step: int, best_acc: float, run_name: str):
    torch.save({
        "phase": phase,                    # "head" or "finetune"
        "epoch": epoch,                    # last completed epoch *within this phase*
        "global_epoch": global_epoch,
        "global_step": global_step,
        "best_acc": best_acc,
        "run_name": run_name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
    }, path)


def load_checkpoint(path: str, device: torch.device) -> dict:
    ckpt = torch.load(path, map_location=device)
    required_keys = {"phase", "epoch", "global_epoch", "global_step", "best_acc", "model_state_dict"}
    missing = required_keys - ckpt.keys()
    if missing:
        raise ValueError(f"Checkpoint at {path} is missing expected keys: {missing}")
    return ckpt

### Training Loop Utilities

MixUp/CutMix (below) is a standard regularizer for exactly the kind of confusion a
confusion-matrix analysis often reveals - visually similar but genuinely distinct classes
(different meat cuts, different pasta dishes, etc.). It trains on deliberately blended images
with soft labels, forcing the model to learn sharper decision boundaries between them instead
of just memorizing "clean" single-class examples.

In [ ]:
def mixup_cutmix(images: torch.Tensor, labels: torch.Tensor, num_classes: int,
                  mixup_alpha: float = 0.2, cutmix_alpha: float = 1.0, prob: float = 0.5):
    """
    Randomly applies MixUp (pixel-blend two images) or CutMix (paste a
    rectangular patch from one image onto another) to a training batch, with
    probability `prob` of applying either at all (chosen 50/50 between the
    two when applying). Returns (images, soft_targets) where soft_targets is
    a (batch_size, num_classes) probability-distribution tensor -- suitable
    for nn.CrossEntropyLoss, which accepts either class-index or
    probability-distribution targets.

    If no mixing is applied this batch, returns the images unchanged and
    ordinary one-hot targets (so downstream loss/accuracy code doesn't need
    to special-case the "no mixing happened" case).
    """
    batch_size = images.size(0)
    targets = F.one_hot(labels, num_classes).float()

    if random.random() > prob:
        return images, targets

    perm = torch.randperm(batch_size, device=images.device)

    if random.random() < 0.5:
        # CutMix: paste a random box from one image onto another.
        lam = float(np.random.beta(cutmix_alpha, cutmix_alpha))
        H, W = images.shape[-2:]
        cut_ratio = (1 - lam) ** 0.5
        cut_w, cut_h = int(W * cut_ratio), int(H * cut_ratio)
        cx, cy = np.random.randint(W), np.random.randint(H)
        x1, x2 = int(np.clip(cx - cut_w // 2, 0, W)), int(np.clip(cx + cut_w // 2, 0, W))
        y1, y2 = int(np.clip(cy - cut_h // 2, 0, H)), int(np.clip(cy + cut_h // 2, 0, H))
        images = images.clone()
        images[:, :, y1:y2, x1:x2] = images[perm, :, y1:y2, x1:x2]

        # Recompute lambda from the ACTUAL pasted area (the box can get
        # clipped at image edges, so the requested and actual areas differ).
        lam = 1.0 - ((x2 - x1) * (y2 - y1) / (W * H))

    else:
        # MixUp: blend two whole images by a random ratio.
        lam = float(np.random.beta(mixup_alpha, mixup_alpha))
        images = lam * images + (1 - lam) * images[perm]

    targets = lam * targets + (1 - lam) * targets[perm]
    return images, targets

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool,
              writer: SummaryWriter = None, global_step: int = 0, grad_clip_norm: float = 1.0):
    """
    One pass over the data. Returns (avg_loss, accuracy, macro_f1, global_step).

    F1 is computed once over the whole epoch's accumulated predictions (not
    averaged per-batch) using macro averaging, so every food class counts
    equally regardless of how visually distinctive or common it is.

    If `writer` is given and train=True, logs every individual batch's loss
    to TensorBoard under "Loss/train_step" -- the ground truth for whether
    the model is learning, since the live progress bar and per-epoch average
    can both be too noisy/coarse to judge a trend from by eye.
    """
    model.train() if train else model.eval()
    if train:
        freeze_batchnorm_stats(model)

    total_loss, total_correct, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        pbar = tqdm(loader, desc="train" if train else "eval", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                # Clips gradient norm so one unlucky/hard batch can't produce
                # an outsized weight update -- a common source of spiky loss.
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                optimizer.step()
                if writer is not None:
                    writer.add_scalar("Loss/train_step", loss.item(), global_step)
                global_step += 1

            batch_size = images.size(0)
            preds = outputs.argmax(dim=1)
            total_loss += loss.item() * batch_size
            total_correct += (preds == labels).sum().item()
            total_samples += batch_size

            all_preds.append(preds.detach().cpu())
            all_labels.append(labels.detach().cpu())

            pbar.set_postfix(loss=total_loss / total_samples, acc=total_correct / total_samples)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return total_loss / total_samples, total_correct / total_samples, f1, global_step

In [ ]:
def train_phase(model, train_loader, val_loader, criterion, optimizer, scheduler,
                 device, epochs, phase_name, best_acc, best_state, checkpoint_path,
                 writer: SummaryWriter, global_epoch: int, global_step: int, run_name: str,
                 start_epoch: int = 1, latest_checkpoint_path: str = None, grad_clip_norm: float = 1.0):
    """
    Runs epochs `start_epoch..epochs` of train+eval, tracking and saving the
    best checkpoint. `start_epoch` > 1 when resuming mid-phase.

    Evaluates against `val_loader` every epoch -- NOT the test set. Model
    selection (which checkpoint counts as "best") happens purely off
    validation accuracy, so the held-out test set stays untouched until one
    final evaluation after all training is done.

    `global_epoch`/`global_step` are running counters so phase 1 and phase 2
    (and a resumed continuation of either) land on one continuous x-axis in
    TensorBoard instead of each phase/resume resetting to 0.
    """
    if start_epoch > epochs:
        print(f"[{phase_name}] already completed {epochs} epochs (resumed past this phase) -- skipping")
        return best_acc, best_state, global_epoch, global_step

    for epoch in range(start_epoch, epochs + 1):
        global_epoch += 1
        start = time.time()
        train_loss, train_acc, train_f1, global_step = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True,
            writer=writer, global_step=global_step, grad_clip_norm=grad_clip_norm,
        )
        val_loss, val_acc, val_f1, _ = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        if scheduler is not None:
            scheduler.step()
        elapsed = time.time() - start

        print(
            f"[{phase_name}] epoch {epoch}/{epochs} "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} "
            f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} "
            f"({elapsed:.1f}s)"
        )

        writer.add_scalar("Loss/train", train_loss, global_epoch)
        writer.add_scalar("Loss/val", val_loss, global_epoch)
        writer.add_scalar("Accuracy/train", train_acc, global_epoch)
        writer.add_scalar("Accuracy/val", val_acc, global_epoch)
        writer.add_scalar("F1/train", train_f1, global_epoch)
        writer.add_scalar("F1/val", val_f1, global_epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], global_epoch)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_path)
            print(f"  -> new best model saved to {checkpoint_path} (val_acc={best_acc:.4f})")

        if latest_checkpoint_path is not None:
            save_checkpoint(
                latest_checkpoint_path, model, optimizer, scheduler,
                phase_name, epoch, global_epoch, global_step, best_acc, run_name,
            )

    return best_acc, best_state, global_epoch, global_step

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CFG.logdir}

### Model Training

Trains on `train_loader`, evaluates on `val_loader` every epoch to decide which checkpoint is
"best", and only evaluates on `test_loader` once, right at the end, using the best checkpoint. The test set is never involved in any decision made during training.

In [ ]:
best_acc, best_state, global_epoch, global_step = 0.0, None, 0, 0
start_epoch_head, start_epoch_finetune = 1, 1
resume_phase = None
ckpt = None

if CFG.resume:
    print(f"Resuming from checkpoint: {CFG.resume}")
    ckpt = load_checkpoint(CFG.resume, device)
    model.load_state_dict(ckpt["model_state_dict"])
    best_acc = ckpt["best_acc"]
    best_state = ckpt["model_state_dict"]  # Initialize best_state from the checkpoint
    global_epoch = ckpt["global_epoch"]
    global_step = ckpt["global_step"]
    resume_phase = ckpt["phase"]
    resume_epoch = ckpt["epoch"]

    if resume_phase == "head":
        start_epoch_head = resume_epoch + 1
    elif resume_phase == "finetune":
        start_epoch_head = CFG.epochs_head + 1  # phase 1 already complete -- skip it
        start_epoch_finetune = resume_epoch + 1
    else:
        raise ValueError(f"Unrecognized phase '{resume_phase}' in checkpoint")

    print(f"  Resuming phase='{resume_phase}', last completed epoch={resume_epoch}, "
          f"best_acc so far={best_acc:.4f}")

# Resolve TensorBoard run name: explicit CFG.run_name > resumed checkpoint's
# original run_name (keeps curves continuous) > a fresh timestamp.
if CFG.run_name:
    run_name = CFG.run_name
elif ckpt is not None and ckpt.get("run_name"):
    run_name = ckpt["run_name"]
else:
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

log_dir = str(Path(CFG.logdir) / run_name)
writer = SummaryWriter(log_dir=log_dir)
is_resuming_same_run = bool(CFG.resume) and not CFG.run_name and ckpt is not None and ckpt.get("run_name")
print(f"TensorBoard logging to: {log_dir}" + (" (continuing existing run)" if is_resuming_same_run else ""))

if not is_resuming_same_run:
    hparams_text = "\n".join(f"{k}: {v}" for k, v in CFG.__dict__.items())
    writer.add_text("hyperparameters", hparams_text)
    sample_images, _ = next(iter(train_loader))
    writer.add_graph(model, sample_images.to(device))

# ---------- Phase 1: train head only ----------
print("\n=== Phase 1: training classifier head (backbone frozen) ===")
set_backbone_trainable(model, trainable=False)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.lr_head, weight_decay=CFG.weight_decay,
)
scheduler = build_scheduler(optimizer, CFG.epochs_head, CFG.warmup_epochs)
if CFG.resume and resume_phase == "head":
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if ckpt["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
best_acc, best_state, global_epoch, global_step = train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, CFG.epochs_head, "head", best_acc, best_state, checkpoint_path,
    writer, global_epoch, global_step, run_name,
    start_epoch=start_epoch_head, latest_checkpoint_path=latest_checkpoint_path,
    grad_clip_norm=CFG.grad_clip_norm,
)

# ---------- Phase 2: fine-tune (partially unfrozen backbone) ----------
print("\n=== Phase 2: fine-tuning from", CFG.finetune_unfreeze_from, "onward ===")
set_layers_trainable(model, unfreeze_from=CFG.finetune_unfreeze_from)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.lr_finetune, weight_decay=CFG.weight_decay,
)
scheduler = build_scheduler(optimizer, CFG.epochs_finetune, CFG.warmup_epochs)
if CFG.resume and resume_phase == "finetune":
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if ckpt["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

mixup_fn = None
if CFG.use_mixup:
    mixup_fn = lambda images, labels: mixup_cutmix(
        images, labels, NUM_CLASSES,
        mixup_alpha=CFG.mixup_alpha, cutmix_alpha=CFG.cutmix_alpha, prob=CFG.mixup_prob,
    )

best_acc, best_state, global_epoch, global_step = train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, CFG.epochs_finetune, "finetune", best_acc, best_state, checkpoint_path,
    writer, global_epoch, global_step, run_name,
    start_epoch=start_epoch_finetune, latest_checkpoint_path=latest_checkpoint_path,
    grad_clip_norm=CFG.grad_clip_norm, mixup_fn=mixup_fn,
)

# ---------- Final, one-time evaluation on the held-out test set ----------
# Loads the BEST checkpoint (highest val accuracy) rather than whatever the
# last epoch happened to produce, and touches the test set exactly once,
# now that every training/model-selection decision is already finalized.
print("\n=== Final evaluation on held-out test set ===")
model.load_state_dict(best_state)
test_loss, test_acc, test_f1, _ = run_epoch(model, test_loader, criterion, None, device, train=False)
print(f"Test accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f} | Test loss: {test_loss:.4f}")
writer.add_scalar("Accuracy/test_final", test_acc, global_epoch)
writer.add_scalar("F1/test_final", test_f1, global_epoch)

writer.add_hparams(
    {k: v for k, v in CFG.__dict__.items() if isinstance(v, (int, float, str))},
    {"best_val_accuracy": best_acc, "final_test_accuracy": test_acc},
)
writer.close()

print(f"\nTraining complete. Best val accuracy: {best_acc:.4f} | Final test accuracy: {test_acc:.4f}")
print(f"Best weights saved to: {checkpoint_path}")

## Diagnostics: confusion matrix & per-class accuracy

Evaluated against the **validation set** (not test -- that stays untouched, reserved for the
one final unbiased number already computed above). With 315 classes, a raw confusion matrix
heatmap is too dense to read cell-by-cell -- individual axis labels aren't legible at this
scale, and that's expected, not a bug. It's still useful to eyeball overall structure (a clean
diagonal vs. visible off-diagonal blocks of systematically confused classes). The two tables
below it are the more directly actionable outputs: which specific classes are weakest, whether
that's explained by them simply having less training data, and which specific pairs of dishes
the model actually confuses for each other.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="predicting"):
        images = images.to(device, non_blocking=True)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


# Diagnose the same BEST checkpoint used for the final test score, evaluated
# on val -- val is fine to inspect repeatedly since (unlike test) it isn't
# meant to give one final unbiased number, it's meant to be looked at.
model.load_state_dict(best_state)
val_preds, val_labels = collect_predictions(model, val_loader, device)

cm = confusion_matrix(val_labels, val_preds, labels=range(NUM_CLASSES))
support = cm.sum(axis=1)  # true val examples per class

# --- Full confusion matrix as a density plot. Log-scaled so the large
# diagonal counts don't wash out the much smaller (but more interesting)
# off-diagonal ones. No per-class tick labels -- 315 of them would be
# unreadable; the point here is overall shape, not individual cells.
plt.figure(figsize=(12, 12))
plt.imshow(np.log1p(cm), cmap="viridis")
plt.title(f"Validation confusion matrix ({NUM_CLASSES} classes, log-scaled)")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.colorbar(label="log(count + 1)")
plt.tight_layout()
plt.show()

# --- Per-class accuracy, correlated against how much TRAIN data each class
# actually had -- directly tests whether the weakest classes are simply
# data-starved (a case FOR more targeted data) or something else (a case
# for a different fix, e.g. regularization or genuinely confusable dishes).
train_class_counts = defaultdict(int)
for _, label in train_loader.dataset.samples:
    train_class_counts[label] += 1

rows = []
for idx, class_name in enumerate(class_names):
    if support[idx] == 0:
        continue  # no val examples for this class -- can't measure its accuracy
    rows.append({
        "class": class_name,
        "val_accuracy": cm[idx, idx] / support[idx],
        "val_support": int(support[idx]),
        "train_support": train_class_counts.get(idx, 0),
    })

results_df = pd.DataFrame(rows).sort_values("val_accuracy")
n_zero_val = NUM_CLASSES - len(rows)
if n_zero_val:
    print(f"{n_zero_val} classes have zero validation examples (too few training images to "
          f"split) -- excluded from the table below.\n")

print("Worst 25 classes by validation accuracy:")
display(results_df.head(25))

In [ ]:
correlation = results_df["train_support"].corr(results_df["val_accuracy"])
print(f"\nCorrelation between train-set size and val accuracy: {correlation:.3f}")
print("(closer to +1 = weak classes are mostly explained by having less training data -- "
      "more data would help; closer to 0 = they aren\'t -- look at the confused pairs below instead)")

In [ ]:
# --- Top confused class PAIRS -- the most directly interpretable view of
# "how does this happen": specifically which dishes get mixed up with which.
pairs = []
for true_idx in range(NUM_CLASSES):
    for pred_idx in range(NUM_CLASSES):
        if true_idx != pred_idx and cm[true_idx, pred_idx] > 0:
            pairs.append((int(cm[true_idx, pred_idx]), class_names[true_idx], class_names[pred_idx]))
pairs.sort(key=lambda p: p[0], reverse=True)

print("\nTop 25 most confused class pairs (true -> predicted : count):")
for count, true_name, pred_name in pairs[:25]:
    print(f"  {true_name:30s} -> {pred_name:30s} : {count}")